# Working with SQLite3

- SQLite: Lightweight, serverless, embedded SQL database engine implemented as a single library. Database is a single file on disk.

- sqlite3 (Python stdlib module): Built-in Python API that embeds SQLite; provides Connection, Cursor, transaction controls, and parameterized queries.

- ACID (in SQLite context): Atomicity, Consistency, Isolation, Durability — SQLite provides - ACID-compliant transactions by default (with caveats related to journaling modes).

- Transactions: Grouping multiple operations inside BEGIN/COMMIT/ROLLBACK. In Python: connection’s commit/rollback or context-manager pattern.

- WAL (Write-Ahead Logging): Journaling mode that improves concurrency and performance for reads/writes; alternative to default rollback journal.

- BLOB: Binary Large OBject, used to store binary data (images, blobs) inside tables — supported but use with care.

- Parameterized queries: Using ? or named parameters to prevent SQL injection and improve performance.

- Row factory: Mechanism to control how query results are returned (tuples, dict-like objects, namedtuple).

- SQLite limitations: Single file locks for writes (limited concurrent writers), limited user-defined function performance, not optimized for high-concurrency heavy write workloads.

In [ ]:
import sqlite3

| **Concept**           | **Analogy**              | **Description**                                                                 |
|------------------------|--------------------------|---------------------------------------------------------------------------------|
| Database Connection    | The base or launcher     | It’s the structure holding everything together — the connection to the SQLite database file. |
| Cursor                 | The catapult arm         | It’s what launches (executes) commands toward the database. You use it to send SQL instructions. |
| SQL Command            | The payload (stone)      | It’s what you want to deliver — an INSERT, SELECT, or any SQL statement.        |
| Database Engine        | The target area          | It’s where the payload lands — the database engine receives and processes the command. |
| Result Set             | The returned impact or splash | After the payload hits, you get data back — fetched rows, confirmation, etc. |



- You build the catapult base → conn = sqlite3.connect('data.db')
- You arm the catapult → cursor = conn.cursor()
- You load a payload (SQL command) → cursor.execute("SELECT * FROM users")
- You launch it → the database executes the SQL
- You inspect the results of the hit → rows = cursor.fetchall()

In [ ]:
#connecting to a sqlite database
conn = sqlite3.connect('example.db')
conn

In [ ]:
cur=conn.cursor()

In [ ]:
#create a table:
cur.execute('''
    create table if not exists employees(
                id Integer Primary Key,
                name Text Not Null,
                age Integer,
                dept Text
            )
''')
#commit the changes:
conn.commit()

In [ ]:
cur.execute('''
    select * from employees
''')

In [ ]:
#inserting the data:
cur.execute('''
    insert into employees(name, age, dept)
            values('Vanitas',21,'Doctor')
''')
cur.execute('''
    insert into employees(name, age, dept)
            values('Noe',29,'Reader')
''')
cur.execute('''
    insert into employees(name, age, dept)
            values('Gilbert',27,'Butler')
''')

conn.commit()

In [ ]:
#quering the data
cur.execute('select * from employees')
rows = cur.fetchall()

#printing them
for row in rows:
    print(row)

In [ ]:
#update the data:
cur.execute('''
    update employees set dept='Reader'
            where id=2
''')
conn.commit()

In [ ]:
#querying the data
cur.execute('select * from employees')
rows = cur.fetchall()

#printing them
for row in rows:
    print(row)

In [ ]:
#delete the date from the table
cur.execute('''
    delete from employees where id>3
''')

conn.commit()

In [ ]:
#querying the data
cur.execute('select * from employees')
rows = cur.fetchall()

#printing them
for row in rows:
    print(row)

In [ ]:
# redoing it all as to show a complete process:
import sqlite3 as sq3

In [ ]:
connection = sq3.connect('sales_data.db')
cursor = connection.cursor()

#creating the table:
cursor.execute('''
    create table if not exists sales(
        id Integer Primary Key,
        date Text not null,
        product text not null,
        sales Integer,
        region Text
    )
''')

#inserting data:
sales_data = [
    ('2023-01-01', 'Product1', 100, 'North'),
    ('2023-01-02', 'Product2', 200, 'South'),
    ('2023-01-03', 'Product1', 150, 'East'),
    ('2023-01-04', 'Product3', 250, 'West'),
    ('2023-01-05', 'Product2', 300, 'North')
]

#to insert all these at once:
cursor.executemany('''
    insert into sales(date,product,sales,region) values(?,?,?,?)
''',sales_data)

connection.commit()

In [ ]:
#quering the data
cursor.execute('select * from sales')
rows = cursor.fetchall()

#printing them
for row in rows:
    print(row)

In [ ]:
connection.close()

i will now be adding some snippets i got from gpt

In [ ]:
import sqlite3

# 1. Connect to the database (or create one)
conn = sqlite3.connect('students.db')

# 2. Create a cursor object
cursor = conn.cursor()

# 3. Execute SQL commands
cursor.execute('CREATE TABLE IF NOT EXISTS students (id INTEGER PRIMARY KEY, name TEXT, grade REAL)')
cursor.execute('INSERT INTO students (name, grade) VALUES (?, ?)', ('Alice', 89.5))
cursor.execute('INSERT INTO students (name, grade) VALUES (?, ?)', ('Bob', 92.0))

# 4. Commit changes
conn.commit()

# 5. Retrieve data
cursor.execute('SELECT * FROM students')

# 6. Fetch results
rows = cursor.fetchall()
for row in rows:
    print(row)

# 7. Close the cursor and connection
cursor.close()
conn.close()


In [ ]:
import sqlite3

# 1️⃣ Connect to (or create) a database file
conn = sqlite3.connect("people.db")

# 2️⃣ Create a cursor object
cur = conn.cursor()

# 3️⃣ Create a table if it doesn't exist yet
cur.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    age INTEGER
)
""")

# 4️⃣ Insert some sample data
cur.executemany("INSERT INTO users (name, age) VALUES (?, ?)", [
    ("Alice", 23),
    ("Bob", 31),
    ("Charlie", 28),
])
conn.commit()

# 5️⃣ Take input from the user
min_age = int(input("Enter minimum age: "))

# 6️⃣ Use a parameterized query with a tuple
cur.execute("SELECT name, age FROM users WHERE age > ?", (min_age,))

# 7️⃣ Fetch and display results
rows = cur.fetchall()
print("\nUsers older than", min_age, ":")
for name, age in rows:
    print(f" - {name} ({age})")

# 8️⃣ Clean up
cur.close()
conn.close()


In [ ]:
import sqlite3

# Context-managed connection ensures commit/rollback
with sqlite3.connect("data.db") as conn:
    cur = conn.cursor()
    cur.execute("CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT, age INTEGER)")
    cur.execute("INSERT INTO users (name, age) VALUES (?, ?)", ("Alice", 30))
    # conn.commit() is automatic on exiting 'with' if no exception


In [ ]:
cur.execute("SELECT * FROM users WHERE age > ?", (25,))
rows = cur.fetchall()


In [ ]:
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("SELECT * FROM users")
rows = cur.fetchall()

for row in rows:
    print(row["id"], row["name"], row["age"]) 


In [ ]:
conn = sqlite3.connect("data.db")
try:
    conn.execute("BEGIN")
    # multiple statements
    conn.execute(...)
    conn.commit()
except Exception:
    conn.rollback()
    raise
finally:
    conn.close()


In [ ]:
import pandas as pd
df = pd.read_sql_query("SELECT * FROM users", conn)
df

# Logging in Python

### Python Logging Concepts

| **Concept** | **Description** |
|-------------|----------------|
| Logging     | The process of recording diagnostic, audit, and runtime information about a program’s execution — used for debugging, monitoring, and post-mortem analysis. |
| Logger      | The interface through which you issue log statements (`logging.getLogger()` object). |
| Handler     | Defines where the log messages go (console, file, network, etc.). |
| Formatter   | Defines the log message layout (timestamp, module name, log level, message). |
| Level       | Defines the severity threshold that determines which messages are handled. |

#### Common Logging Levels

| **Level**   | **Numeric** | **Purpose** |
|------------|------------|-------------|
| DEBUG      | 10         | Detailed info for developers |
| INFO       | 20         | Normal operation messages |
| WARNING    | 30         | Something unexpected, but still working |
| ERROR      | 40         | Serious issue, operation failed |
| CRITICAL   | 50         | Application failure / system outage |


In [ ]:
import logging

In [ ]:
#configure the basic logging settings
logging.basicConfig(level=logging.DEBUG)

#log msg:
logging.debug('Debug msg')
logging.warning('warning msg')
logging.error('error msg')
logging.critical('critical msg')
logging.info('info msg')

In [ ]:
#configuring the logging
#mind u if a logging config has been set we need to restart the terminal
#to intruduce a new one

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s-%(name)s-%(levelname)s-%(message)s',
    datefmt='%Y-%m-%d ~ %H:%M:%S'
)
#log msg with different severity levels:
logging.debug('Debug msg')
logging.warning('warning msg')
logging.error('error msg')
logging.critical('critical msg')
logging.info('info msg')

This is how the output would be if we dont restart the kernel

```
DEBUG:root:Debug msg
WARNING:root:warning msg
ERROR:root:error msg
CRITICAL:root:critical msg
INFO:root:info msg
```


In [ ]:
#now we store the msgs in a file 
#yes we have to always have a new kernel or at least one
#where we haven't setup the basicConfig else the old settings will
#take precedence

logging.basicConfig(
    filename='log1_trial.log',
    filemode='w',
    level=logging.DEBUG,
    format='%(asctime)s-%(name)s-%(levelname)s-%(message)s',
    datefmt='%Y-%m-%d ~ %H:%M:%S'
)
#log msg with different severity levels:
logging.debug('Debug msg')
logging.warning('warning msg')
logging.error('error msg')
logging.critical('critical msg')
logging.info('info msg')

In [ ]:
logging.debug('Debug msg')
logging.warning('warning msg')
logging.error('error msg')
logging.critical('critical msg')
logging.info('info msg')

In [ ]:
logging.info('now i am just messing around here')

### logging with multiple loggers

Using `logging.getLogger()` in Python Modules

1️⃣ What is `getLogger()`?

```python
import logging

logger = logging.getLogger("mylogger")
logger.info("This is a log message")
```

Returns a Logger object with the given name.

- If the logger already exists, it returns the same object (singleton per name).  
- If no name is given, it returns the root logger.

---

2️⃣ Using Named Loggers in Modules

For a project with multiple modules:

```python
# main.py
import logging
logger = logging.getLogger("main")
logger.info("Main process started")

# utils.py
import logging
logger = logging.getLogger("utils")
logger.debug("Utility function called")
```

**Benefits:**

- Identifies which module the log came from  
- Allows hierarchical loggers like `"service.database"`

---

3️⃣ Recommended Pattern: Use `__name__`

```python
# In each module
import logging
logger = logging.getLogger(__name__)
logger.info("Module-specific message")
```

- `__name__` automatically sets the logger name to the module name.  
- Works well with hierarchical logging.

Example output:

```
2025-10-29 14:30:00 - service.database - INFO - DB connected
2025-10-29 14:30:01 - utils - DEBUG - Utility function called
```

---

4️⃣ Summary

| Feature               | Benefit |
|----------------------|---------|
| Named loggers         | Identify which module the log comes from |
| Hierarchical loggers  | Allows parent/child relationship; easy filtering and propagation |
| `__name__` convention | Auto-uses the module name; reduces manual naming errors |
| `.getLogger()` consistency | Same logger object returned for the same name |

💡 Tip: Centralize logging setup in one file (like `logger.py`) and use `logger = logging.getLogger(__name__)` in modules for clean and consistent logging.


In [ ]:
import logging

#creating seperate logger modules:
#Logger module2:
logger1 = logging.getLogger('module1')
logger1.setLevel(logging.DEBUG)

#Logger module2:
logger2 = logging.getLogger('module2')
logger2.setLevel(logging.WARNING)

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s- %(name)s- %(levelname)s -%(message)s',
    datefmt='%Y-%m-%d ~ %H:%M:%S'
)

In [ ]:
#log msg with different loggers
logger1.debug("debug msg for module 1")
logger2.warning("warning msg for module 2")
logger2.error("error msg for module 2")

GPT stuff:

In [ ]:
#Basic logging setup
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.debug("Debugging details")
logging.info("System is running")
logging.warning("Resource usage high")
logging.error("Error occurred")
logging.critical("System failure")

In [ ]:
#Logger hierarchy
logger = logging.getLogger("data.pipeline")
logger.setLevel(logging.DEBUG)
logger.info("Pipeline initialized")

In [ ]:
#File logging
logging.basicConfig(
    filename='app.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)


In [ ]:
#Rotating file handler (production pattern)
from logging.handlers import RotatingFileHandler

handler = RotatingFileHandler("app.log", maxBytes=2000000, backupCount=3)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

logger = logging.getLogger("service")
logger.setLevel(logging.INFO)
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Starting process")
# this basically ensures the logs are of
#sertain size and deletes the old ones

# MultiThreading and Multiprocessing

| Term                              | Definition                                                                                                                           |
| --------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------ |
| **Concurrency**                   | Structuring a program to handle multiple tasks that *overlap in time* but may not literally run simultaneously.                      |
| **Parallelism**                   | Performing multiple computations *at the same physical time* (true simultaneous execution on multiple cores).                        |
| **Thread**                        | Lightweight unit of execution within a single process sharing the same memory space.                                                 |
| **Process**                       | Independent execution unit with its own memory and system resources.                                                                 |
| **GIL (Global Interpreter Lock)** | CPython’s mutex ensuring only one thread executes Python bytecode at a time — prevents true parallel CPU-bound execution in threads. |
| **I/O-bound task**                | Spends time waiting on input/output (network, disk, etc.). Threads excel here.                                                       |
| **CPU-bound task**                | Heavy computation that keeps CPU busy. Multiprocessing excels here.                                                                  |


codes for this is in it's seperate folder! i.e, yes they are below as well but best to try them in the file in the respective folder

### Multi-Threading:

In [ ]:
import threading
import time

In [ ]:
def print_nums():
    for i in range(1,27):
        print(f"Number:  {i}")
        time.sleep(1)

def print_letters():
    for i in range(ord('a'),ord('z')+1):
        print(f'Letters: {chr(i)}')
        time.sleep(1)

t = time.time()
print_nums()
print_letters()
print(time.time()-t)

In [ ]:
#now we running them concurrently using 2 threads:

def print_nums():
    for i in range(1,27):
        print(f"Number:  {i}")
        time.sleep(1)

def print_letters():
    for i in range(ord('a'),ord('z')+1):
        print(f'Letters: {chr(i)}')
        time.sleep(1)


# print_nums()
# print_letters()

#creating 2 threads:
t1= threading.Thread(target=print_nums)
t2= threading.Thread(target=print_letters)

t = time.time()

#starting threads:
t1.start()
t2.start()

#wait for them to complete: 
t1.join()
t2.join()
#now they joined to the main thread

print(time.time()-t)

In [ ]:
import threading
import time

lock = threading.Lock()

def print_nums():
    for i in range(1, 27):
        with lock:
            print(f"Number:  {i}")
        time.sleep(1)

def print_letters():
    for i in range(ord('a'), ord('z') + 1):
        with lock:
            print(f'Letters: {chr(i)}')
        time.sleep(1)

t1 = threading.Thread(target=print_nums)
t2 = threading.Thread(target=print_letters)

t = time.time()
t1.start()
t2.start()
t1.join()
t2.join()

print(time.time() - t)
#Now, only one thread can execute the
#print() at a time, keeping your output tidy.

### Multi-Processing:

In [ ]:
#Processes that run in parallel
#CPU-Bound tasks i.e, computational heavy like (Mathematical computation,
#data processing)
#parallel execution-multiple cores of CPU

import multiprocessing
import time

def square_numbers():
    for i in range(5):
        time.sleep(1)
        print(f"Square: {i*i}")
        
def cube_numbers():
    for i in range(5):
        time.sleep(1)
        print(f"Cube: {i*i*i}")
if __name__=='__main__':
    #creating processes
    p1=multiprocessing.Process(target=square_numbers)
    p2=multiprocessing.Process(target=cube_numbers)
    t=time.time()

    #start the process:
    p1.start()
    p2.start()

    #wait for it to complete:
    p1.join()
    p2.join()

    print(time.time()-t)

### Advance multi-threading

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def print_numbers(num):
    time.sleep(1)
    return f"Number: {num}"
numbers = [1,2,3,4,5,6,7,8]

with ThreadPoolExecutor(max_workers=3) as executor:
    results=executor.map(print_numbers,numbers)

for result in results:
    print(result)

In [ ]:
'''
Real-World Example: Multithreading for I/O-bound Tasks
Scenario: Web Scraping 
Web scraping often involves making numerous 
network requests to fetch web pages. 
These tasks are I/O-bound because they spend a lot of time waiting 
for responses from servers. Multithreading can significantly improve
the performance by allowing multiple web pages to be fetched concurrently.
'''

'''
https://www.langchain.com/
https://docs.langchain.com/oss/python/langchain/overview?_gl=1*mp1jnt*_ga*MTAwNTEyNjA3My4xNzYxNzY3NzEz*_ga_47WX3HKKY2*czE3NjE3Njc3MTMkbzEkZzAkdDE3NjE3Njc3MTMkajYwJGwwJGgw
https://docs.langchain.com/oss/python/langchain/tools
https://docs.langchain.com/oss/python/langchain/agents
'''

import threading
import requests
from bs4 import BeautifulSoup
import re

urls=[
    'https://www.langchain.com/',
    'https://docs.langchain.com/oss/python/langchain/overview?_gl=1*mp1jnt*_ga*MTAwNTEyNjA3My4xNzYxNzY3NzEz*_ga_47WX3HKKY2*czE3NjE3Njc3MTMkbzEkZzAkdDE3NjE3Njc3MTMkajYwJGwwJGgw;',
    'https://docs.langchain.com/oss/python/langchain/tools',
    'https://docs.langchain.com/oss/python/langchain/agents'
]


def sanitize_filename(url):
    # Replace all characters that are NOT letters, numbers, _, or - with _
    return re.sub(r'[^a-zA-Z0-9_-]', '_', url)

def fetch_content(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    print(f'Fetched {len(soup.get_text())} characters from {url}')

    filename = f"file_{sanitize_filename(url)}.txt"
    with open(filename, 'a', encoding='utf-8') as file:
        file.write(url + "\n\n\n")
        file.write(soup.get_text())

threads = []

for url in urls:
    thread = threading.Thread(target=fetch_content,args = (url,))
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

print("all the urls are fetched")


### Advance multi-processing

In [ ]:
#wont work here cuz its a notebook
from concurrent.futures import ProcessPoolExecutor
import time

def square_numbers(num):
    time.sleep(1)
    return f"Square: {num*num}"
numbers = [1,2,3,4,5,6,7,8]

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=3) as executor:
        results=executor.map(square_numbers,numbers)

    for result in results:
        print(result)

| Feature             | Basic `Process`             | `ProcessPoolExecutor`                                  |
| ------------------- | --------------------------- | ------------------------------------------------------ |
| Number of processes | You create manually         | You specify pool size; tasks distributed automatically |
| Reuse processes     | No, new process each time   | Yes, processes are reused                              |
| Ease of scaling     | Harder for many tasks       | Easy with `.map()` or `.submit()`                      |
| Overhead            | Higher for many short tasks | Lower because of reuse                                 |


| Aspect            | Threading                        | Multiprocessing                     |
| ----------------- | -------------------------------- | ----------------------------------- |
| Execution model   | Multiple threads in one process  | Separate processes                  |
| Memory space      | Shared                           | Isolated                            |
| Best for          | I/O-bound tasks                  | CPU-bound tasks                     |
| GIL impact        | Present (limits CPU parallelism) | Bypassed (each process has own GIL) |
| Overhead          | Low                              | High (process spawn cost)           |
| Data sharing      | Simple (shared memory)           | Complex (serialization needed)      |
| Failure isolation | Weak                             | Strong (process crash isolated)     |


| Topic           | Library                                  | Key Classes/Functions                           |
| --------------- | ---------------------------------------- | ----------------------------------------------- |
| Threading       | `threading`                              | `Thread`, `Lock`, `RLock`, `Event`, `Semaphore` |
| Thread pools    | `concurrent.futures`                     | `ThreadPoolExecutor`                            |
| Multiprocessing | `multiprocessing`                        | `Process`, `Queue`, `Pipe`, `Manager`           |
| Process pools   | `multiprocessing` / `concurrent.futures` | `Pool`, `ProcessPoolExecutor`                   |
| Communication   | `multiprocessing`                        | `Queue`, `Manager`, `Value`, `Array`            |
| Parallel map    | Both                                     | `executor.map()` / `pool.map()`                 |


# Python Memory Management

Yes me just watched the video not gonna code as it was mostly gc
and memory efficiency  techniques like using generators aka yield and such

so re-watch

| Concept                     | Definition                                                                                                    |
| --------------------------- | ------------------------------------------------------------------------------------------------------------- |
| **Memory Management**       | The process by which Python allocates, tracks, and reclaims memory used by objects during runtime.            |
| **Reference**               | A variable name or container entry pointing to an object’s memory address.                                    |
| **Reference Count**         | Number of references pointing to a given object. When it drops to zero, the object’s memory is reclaimed.     |
| **Garbage Collection (GC)** | Automatic memory reclamation system that detects and clears unreferenced or cyclic objects.                   |
| **Heap**                    | Memory region from which Python allocates all objects (everything lives on the heap).                         |
| **Stack**                   | Used internally for call frames, local variables, and function execution context.                             |
| **Object Interning**        | Optimization where immutable small objects (like small integers and strings) are reused instead of recreated. |
| **Memory Leak**             | Situation where objects remain referenced (often unintentionally), preventing GC from reclaiming memory.      |


| Goal               | Practice                                           |
| ------------------ | -------------------------------------------------- |
| Reduce footprint   | Use generators, iterators, NumPy arrays.           |
| Avoid leaks        | Break reference cycles, manage globals carefully.  |
| Improve efficiency | Use `__slots__`, reuse objects.                    |
| Debug usage        | Use `tracemalloc`, `memory_profiler`, `gc` module. |
| Prevent overhead   | Avoid deep recursion, excessive object creation.   |


In [ ]:
# [FreeCourseSite.com] Udemy - The Complete Full-Stack Web Development Bootcamp